# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.


In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DataWithHamza/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print(os.getcwd())

/content/Flyrank-ML-Internship


## 1. Ranked actions + reason codes


The ranked queue is designed to help a content team decide which pages
should be reviewed first when review capacity is limited.

Each page receives a recommendation based on observed search and
engagement signals.

Reason codes used in this notebook:

- LOW_CTR – page receives impressions but relatively few clicks.
- POOR_POSITION – average search position is weak.
- LOW_ENGAGEMENT – visitors engage less than expected.
- DECLINING_TRAFFIC – recent traffic is lower than historical performance.
- OLD_CONTENT – page has not been updated for a long time.

The queue is intended to support human review rather than automatically
change or publish content.

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Apply the same filters used throughout your project
df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

df = df.drop_duplicates(subset="content_id")

print(df.shape)

(30000, 44)


In [4]:
import pandas as pd
import numpy as np

queue = df.copy()

queue["reason_code"] = ""

queue.loc[queue["ctr"] < queue["ctr"].median(), "reason_code"] += "LOW_CTR; "

queue.loc[
    queue["avg_position"] > queue["avg_position"].median(),
    "reason_code"
] += "POOR_POSITION; "

queue.loc[
    queue["engagement_rate"] < queue["engagement_rate"].median(),
    "reason_code"
] += "LOW_ENGAGEMENT; "

queue.loc[
    queue["trend_direction"] == "down",
    "reason_code"
] += "DECLINING_TRAFFIC; "

queue.loc[
    queue["days_since_last_update"] >
    queue["days_since_last_update"].median(),
    "reason_code"
] += "OLD_CONTENT"

# Priority score
queue["priority_score"] = (
    (1 - queue["ctr"].rank(pct=True))
    + queue["avg_position"].rank(pct=True)
    + (1 - queue["engagement_rate"].rank(pct=True))
)

queue = queue.sort_values(
    "priority_score",
    ascending=False
)

display(
    queue[
        [
            "content_id",
            "priority_score",
            "reason_code"
        ]
    ].head(20)
)

,content_id,priority_score,reason_code
24445,content_661e1745db72,2.419283,LOW_CTR; POOR_POSITION;
19920,content_23f1cc8851a9,2.419250,LOW_CTR; POOR_POSITION; OLD_CONTENT
26873,content_7275a6a3a8eb,2.419217,LOW_CTR; POOR_POSITION; OLD_CONTENT
16044,content_71a31b831092,2.419183,LOW_CTR; POOR_POSITION; OLD_CONTENT
18532,content_42c7c72b8391,2.419150,LOW_CTR; POOR_POSITION;
15639,content_cb6c7d58c0bc,2.419117,LOW_CTR; POOR_POSITION;
27923,content_692fda8c52bd,2.419083,LOW_CTR; POOR_POSITION; OLD_CONTENT
2970,content_3e087a5d8f15,2.419050,LOW_CTR; POOR_POSITION;
28214,content_13bbd72aea33,2.419017,LOW_CTR; POOR_POSITION; DECLINING_TRAFFIC; OLD...
1534,content_abeb1aa40158,2.418983,LOW_CTR; POOR_POSITION;


## 2. Intended use and limits


This playbook is intended for SEO specialists and content reviewers who
must decide which pages to inspect first.

The ranked queue helps prioritize limited review capacity by identifying
pages with observed warning signals.

The recommendations should not be interpreted as proof that refreshing a
page will improve rankings or traffic. External factors such as search
algorithm updates, competitor activity, and content quality are not fully
represented in this dataset.

In [5]:
summary = pd.DataFrame({

    "User":[
        "Content Strategist",
        "SEO Specialist"
    ],

    "Primary Use":[
        "Prioritize pages for manual review",
        "Identify pages needing investigation"
    ],

    "Limitation":[
        "Decision-support only",
        "Cannot demonstrate causal impact"
    ]

})

display(summary)

,User,Primary Use,Limitation
0,Content Strategist,Prioritize pages for manual review,Decision-support only
1,SEO Specialist,Identify pages needing investigation,Cannot demonstrate causal impact


## 3. Human review + the no-go list


Before acting on a recommendation, a reviewer should check:

- whether the content is still accurate,
- whether search intent has changed,
- whether competitors provide stronger content,
- whether recent business changes explain the observed trend.

The following actions should never be fully automated:

- publishing new content,
- deleting pages,
- changing important business information,
- making SEO decisions without human review.

The ranked queue is intended to assist expert judgement rather than
replace it.

In [6]:
review = pd.DataFrame({

    "Human Check":[
        "Content accuracy",
        "Search intent",
        "Competitor comparison",
        "Business context"
    ],

    "Required":[
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]

})

display(review)

print("\nNo-Go List")

no_go = [

    "Automatically publish content",

    "Automatically delete pages",

    "Automatically rewrite pages",

    "Automatically change metadata"

]

for item in no_go:
    print("-", item)

,Human Check,Required
0,Content accuracy,Yes
1,Search intent,Yes
2,Competitor comparison,Yes
3,Business context,Yes



No-Go List
- Automatically publish content
- Automatically delete pages
- Automatically rewrite pages
- Automatically change metadata


## 4. Monitoring / retrain triggers


The model should be monitored regularly after deployment.

Possible retraining triggers include:

- Precision@50 decreases over time.
- Search behaviour changes substantially.
- New content types appear.
- Large search engine algorithm updates occur.
- The distribution of important features changes noticeably.

Regular monitoring helps keep the recommendations relevant.

In [7]:
triggers = pd.DataFrame({

    "Trigger":[
        "Precision@50 drops",
        "Traffic distribution changes",
        "Major Google update",
        "New content categories",
        "Feature drift detected"
    ],

    "Action":[
        "Retrain model",
        "Review feature engineering",
        "Re-evaluate model",
        "Update feature set",
        "Retrain and validate"
    ]

})

display(triggers)

,Trigger,Action
0,Precision@50 drops,Retrain model
1,Traffic distribution changes,Review feature engineering
2,Major Google update,Re-evaluate model
3,New content categories,Update feature set
4,Feature drift detected,Retrain and validate


## 5. Exports for the paper


The ranked queue generated in this notebook is exported to the
`work/outputs/` directory.

These exported files can be reused directly in the final capstone paper
to demonstrate the model's recommendations.

The exported queue contains only anonymized identifiers and observed
signals.

In [8]:
from pathlib import Path
import matplotlib.pyplot as plt

# Create output directory
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export top 100 recommendations
export_columns = [
    "content_id",
    "priority_score",
    "reason_code",
    "ctr",
    "avg_position",
    "engagement_rate"
]

queue.head(100)[export_columns].to_csv(
    output_dir / "content_action_queue.csv",
    index=False
)

# Export summary chart
plt.figure(figsize=(8,5))

queue["reason_code"].str.split(";").explode().value_counts().head(10).plot(kind="bar")

plt.title("Top Recommendation Reasons")

plt.tight_layout()

plt.savefig(
    output_dir / "reason_code_frequency.png",
    dpi=300
)

plt.close()

print("Files exported successfully.")

print(output_dir / "content_action_queue.csv")

print(output_dir / "reason_code_frequency.png")

Files exported successfully.
work/outputs/content_action_queue.csv
work/outputs/reason_code_frequency.png
